# Advanced Unsupervised and Semi-Supervised Learning
## Part 1 — Research and Theory

**Prepared by:** Aminah Zareen  
**Internship Task:** Advanced Unsupervised & Semi-Supervised Learning, Clustering, Anomaly Detection, and Model Interpretation

---

### Purpose of this notebook

This notebook develops the theory required in Part 1 of the assignment. I have explained each idea in my own words and connected the mathematical notation to its practical meaning. Every worked example marked **Self-constructed numerical example** was created specifically for this notebook and is not copied from a textbook or dataset.

### Notation used

- $x_i\in\mathbb{R}^d$ denotes a sample with $d$ features.
- $y_i$ denotes a class label when one is available.
- $X_L$ and $Y_L$ denote labeled data, while $X_U$ denotes unlabeled data.
- $\|x\|_2$ is the Euclidean norm.
- $n$ is the number of samples and $k$ is the number of clusters.


# 1.1 Foundations

## 1.1.1 Unsupervised, Semi-Supervised, and Self-Supervised Learning

These three settings differ mainly in where the learning signal comes from.

### Formal distinctions

**Unsupervised learning** receives only feature vectors,

$$
\mathcal{D}_U=\{x_1,x_2,\ldots,x_n\},
$$

and tries to discover structure that was not supplied as a target. A clustering method may minimize an objective such as

$$
\min_{C_1,\ldots,C_k}\sum_{r=1}^{k}\sum_{x_i\in C_r}\|x_i-\mu_r\|_2^2,
$$

where $\mu_r$ is the centre of cluster $C_r$. Dimensionality-reduction methods instead search for a representation $z_i=g(x_i)$ that preserves selected information from $x_i$.

**Semi-supervised learning** uses a small labeled set together with a larger unlabeled set:

$$
\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{l}\cup\{x_j\}_{j=l+1}^{n},\qquad l\ll n.
$$

A common mathematical form combines supervised loss with a regularizer obtained from unlabeled data:

$$
\min_f\;\underbrace{\sum_{i=1}^{l}\ell(f(x_i),y_i)}_{\text{fit the known labels}}
+\lambda\underbrace{\Omega(f;X_L,X_U)}_{\text{respect unlabeled structure}}.
$$

The value of $\lambda$ controls how strongly the unlabeled structure influences the classifier.

**Self-supervised learning** also begins without human labels, but it creates a prediction target from the data itself. If $T$ hides, rotates, or otherwise transforms part of a sample, the model can be trained through

$$
\min_{g,h}\sum_i \ell\big(h(g(T(x_i))),\,t_i\big),
$$

where $t_i$ is generated automatically, such as the hidden word, rotation angle, or missing feature. The representation $g(x)$ is later reused for another task.

### Main assumptions

| Setting | Assumption behind successful learning |
|---|---|
| Unsupervised | A meaningful structure exists and the selected objective or similarity measure can reveal it. |
| Semi-supervised | Unlabeled geometry is related to the unknown labels; nearby or same-cluster samples tend to share a label. |
| Self-supervised | Solving the automatically created task requires features that will also be useful for a later task. |

### Advantages, limitations, and applications

| Setting | Advantages | Limitations | Example applications |
|---|---|---|---|
| Unsupervised | Does not require expensive labels; can reveal unexpected groups or patterns. | A mathematically strong pattern may have no useful domain meaning; evaluation is difficult without labels. | Customer segmentation, topic discovery, exploratory sensor analysis. |
| Semi-supervised | Can achieve better classification with far fewer labels. | Incorrect structural assumptions can make unlabeled data reduce accuracy. | Medical-image classification with a few expert labels, speech recognition, fault classification. |
| Self-supervised | Can learn from very large unlabeled collections and produce reusable representations. | The pretext task may teach shortcuts instead of meaningful structure; training can be expensive. | Masked-language modelling, image representation learning, audio encoders. |

### Self-constructed numerical example

Consider four two-dimensional sensor readings:

$$
x_1=(1.0,1.1),\;x_2=(1.2,0.9),\;x_3=(4.8,5.1),\;x_4=(5.2,4.9).
$$

1. **Unsupervised view:** the distance $d(x_1,x_2)=\sqrt{0.2^2+(-0.2)^2}=0.283$, while $d(x_1,x_3)=\sqrt{3.8^2+4^2}=5.517$. A clustering method can therefore discover the groups $\{x_1,x_2\}$ and $\{x_3,x_4\}$ without labels.
2. **Semi-supervised view:** suppose only $x_1$ is labeled *normal* and $x_3$ is labeled *fault*. The two compact groups provide evidence for assigning *normal* to $x_2$ and *fault* to $x_4$.
3. **Self-supervised view:** hide the second coordinate and train a model to predict it from the first. The automatically available pairs $(1.0\rightarrow1.1)$ and $(4.8\rightarrow5.1)$ provide a learning signal without a person writing class labels.

The same four samples therefore support three different learning settings; what changes is the source and use of the learning signal.


## 1.1.2 Smoothness, Cluster, and Manifold Assumptions

Semi-supervised learning is useful only when unlabeled data tells us something about labels. The following assumptions state when that connection is reasonable.

### Smoothness assumption

If two samples are close in a meaningful representation, their predictions should also be close. On a weighted graph, this can be written as

$$
\Omega_{\text{smooth}}(f)=\frac{1}{2}\sum_{i,j}w_{ij}\|f(x_i)-f(x_j)\|^2,
$$

where a large $w_{ij}$ means that samples $i$ and $j$ are similar. Minimizing this quantity discourages abrupt label changes between strong neighbours.

### Cluster assumption

Samples within the same high-density region are likely to share a label. A good decision boundary should therefore pass through a low-density region rather than split a dense cluster. In probabilistic language, if $x_i$ and $x_j$ belong to the same density component, then $P(Y\mid x_i)$ and $P(Y\mid x_j)$ should be similar.

### Manifold assumption

Although observations may contain many features, their meaningful variation may lie near a lower-dimensional surface $\mathcal{M}\subset\mathbb{R}^d$. Labels are assumed to change smoothly along this surface. For example, hundreds of image pixels may be controlled mainly by a few factors such as pose and lighting.

### Advantages and limitations

| Assumption | Why it helps | When it fails |
|---|---|---|
| Smoothness | Allows labels to spread between similar samples. | Different classes may meet at a sharp boundary, or the selected distance may be misleading. |
| Cluster | Encourages boundaries through sparse areas and can use natural groups. | One cluster may contain several classes, or one class may be split across disconnected clusters. |
| Manifold | Replaces unreliable ambient-space distances with local geometry. | Noise, insufficient sampling, or intersecting manifolds can produce the wrong neighbourhood graph. |

Applications include document classification, activity recognition, medical imaging, and sensor-state identification, where many unlabeled observations are available but labeling is expensive.

### Self-constructed numerical example

Take four points:

$$
A=(0,0),\;B=(0.2,0.1),\;C=(3,3),\;D=(3.2,2.9).
$$

Using $w_{ij}=\exp(-\|x_i-x_j\|^2)$ gives

$$
w_{AB}=e^{-0.05}=0.951,
\qquad
w_{AC}=e^{-18}\approx1.52\times10^{-8}.
$$

Suppose $A$ has label 0 and $C$ has label 1. The strong $A$--$B$ edge and almost nonexistent $A$--$C$ edge support three conclusions:

- **Smoothness:** $B$ should receive a prediction close to the prediction of $A$.
- **Cluster assumption:** $\{A,B\}$ and $\{C,D\}$ form dense pairs separated by a large empty region, so a boundary can be placed between them.
- **Manifold assumption:** if these points were part of two densely sampled curves in a larger feature space, graph paths along each curve would be more meaningful than a direct ambient-space shortcut.

The assumptions are helpful here because geometry and labels agree. If $B$ actually belonged to class 1, the assumptions would be violated and label propagation could confidently make the wrong prediction.


## 1.1.3 Distance and Similarity Metrics

A distance metric $d(x,y)$ should satisfy non-negativity, identity, symmetry, and the triangle inequality. Some useful dissimilarities, especially KL-divergence, do not satisfy all four properties and must not be described as true metrics.

### Euclidean distance

$$
d_2(x,y)=\sqrt{\sum_{j=1}^{d}(x_j-y_j)^2}.
$$

It measures straight-line distance. It assumes that feature scales are comparable and that a spherical notion of closeness is sensible. It is simple and works naturally with K-Means, but it is sensitive to scale, outliers, and high dimensionality. Typical uses include geometric data and standardized continuous measurements.

### Manhattan distance

$$
d_1(x,y)=\sum_{j=1}^{d}|x_j-y_j|.
$$

It measures axis-aligned travel. Compared with Euclidean distance, one very large coordinate difference is not squared, so its influence is less extreme. It can be useful for sparse data and grid-like movement, but it ignores covariance and still requires comparable units.

### Minkowski distance

$$
d_p(x,y)=\left(\sum_{j=1}^{d}|x_j-y_j|^p\right)^{1/p},\qquad p\geq1.
$$

This is a family: $p=1$ gives Manhattan and $p=2$ gives Euclidean distance. Larger $p$ gives more influence to the largest coordinate difference. The flexibility is useful, but selecting $p$ adds a tuning decision. For $0<p<1$, the expression does not satisfy the triangle inequality and is not a metric.

### Cosine similarity and distance

$$
s_{\cos}(x,y)=\frac{x^Ty}{\|x\|_2\|y\|_2},
\qquad d_{\cos}(x,y)=1-s_{\cos}(x,y).
$$

Cosine similarity compares direction rather than magnitude. It assumes that orientation carries more meaning than vector length. It is effective for document vectors and embeddings, but it is undefined for a zero vector and can ignore meaningful magnitude differences. The common quantity $1-s_{\cos}$ is a useful dissimilarity, although it does not satisfy the triangle inequality in every setting.

### Mahalanobis distance

$$
d_M(x,y)=\sqrt{(x-y)^T\Sigma^{-1}(x-y)}.
$$

The covariance matrix $\Sigma$ rescales correlated directions. The method assumes that $\Sigma$ is estimated reliably and is positive definite. It is valuable when features are correlated or have different variances, such as multivariate process monitoring. It becomes unstable when the covariance matrix is singular or poorly estimated, so regularization may be necessary.

### Jaccard similarity and distance

For sets or asymmetric binary features,

$$
J(A,B)=\frac{|A\cap B|}{|A\cup B|},
\qquad d_J(A,B)=1-J(A,B).
$$

Jaccard similarity focuses on shared presences and ignores shared absences. It is useful for baskets of purchased items, symptoms, or tags. It is not suitable when a shared zero is important, and the empty-union case needs an explicit convention.

### Kullback–Leibler divergence

For discrete distributions $P$ and $Q$,

$$
D_{KL}(P\|Q)=\sum_i p_i\log\frac{p_i}{q_i}.
$$

KL-divergence measures the expected information lost when $Q$ represents data generated from $P$. It requires $q_i>0$ whenever $p_i>0$. It is non-negative, but it is asymmetric and does not satisfy the triangle inequality, so it is a divergence rather than a distance metric. It is widely used to compare probability distributions and in variational learning.

### Wasserstein distance

For distributions $P$ and $Q$ over a space with ground cost $c(x,y)$,

$$
W(P,Q)=\inf_{\gamma\in\Pi(P,Q)}\int c(x,y)\,d\gamma(x,y),
$$

where $\Pi(P,Q)$ is the set of transport plans having marginals $P$ and $Q$. The value is the minimum work needed to move probability mass from one distribution to the other. Unlike KL-divergence, it remains meaningful for distributions with non-overlapping support. Its limitation is greater computational cost and dependence on the ground metric.

### Summary of assumptions and suitable uses

| Measure | Main assumption | Strong point | Main limitation | Suitable example |
|---|---|---|---|---|
| Euclidean | Comparable numeric scales | Direct geometric meaning | Outlier and scale sensitivity | Standardized sensor vectors |
| Manhattan | Coordinate-wise changes are meaningful | Less domination by a single large difference | Ignores correlation | Sparse numeric vectors |
| Minkowski | A meaningful value of $p$ exists | General distance family | $p$ must be justified | Tunable nearest-neighbour models |
| Cosine | Direction matters more than length | Effective in sparse high dimensions | Loses magnitude information | Text vectors |
| Mahalanobis | Covariance estimate is reliable | Accounts for scale and correlation | Unstable inverse covariance | Process monitoring |
| Jaccard | Positive matches matter most | Natural for sets/binary presence | Ignores joint zeros | Tags or shopping baskets |
| KL-divergence | Comparable probability distributions | Information-theoretic meaning | Asymmetric; infinite with unsupported events | Comparing class distributions |
| Wasserstein | A meaningful transport cost exists | Uses geometry between supports | Computationally heavier | Comparing shifted distributions |

### Self-constructed numerical examples

Let $x=(1,2)$ and $y=(4,6)$.

$$
d_2(x,y)=\sqrt{3^2+4^2}=5,
$$

$$
d_1(x,y)=3+4=7,
$$

$$
d_3(x,y)=(3^3+4^3)^{1/3}=91^{1/3}\approx4.498,
$$

and

$$
s_{\cos}(x,y)=\frac{1(4)+2(6)}{\sqrt{1^2+2^2}\sqrt{4^2+6^2}}
=\frac{16}{\sqrt{260}}\approx0.9923.
$$

For a Mahalanobis example, let $\Sigma=\operatorname{diag}(9,4)$. Then

$$
d_M(x,y)=\sqrt{\frac{3^2}{9}+\frac{4^2}{4}}=\sqrt{5}\approx2.236.
$$

The same coordinate differences look smaller because variation of that size is expected under the selected covariance.

For sets $A=\{a,b,c\}$ and $B=\{b,c,d,e\}$,

$$
J(A,B)=\frac{2}{5}=0.4,\qquad d_J(A,B)=0.6.
$$

For $P=(0.75,0.25)$ and $Q=(0.5,0.5)$,

$$
D_{KL}(P\|Q)=0.75\ln(1.5)+0.25\ln(0.5)\approx0.1308\text{ nats}.
$$

Finally, suppose two equally weighted observations are located at $P:\{0,2\}$ and $Q:\{1,5\}$. In one dimension, optimal transport pairs the sorted observations:

$$
W_1(P,Q)=\frac{|0-1|+|2-5|}{2}=2.
$$

These examples show why the choice of measure can change the numerical meaning of “close.”


## 1.1.4 Feature Scaling and Its Effect on Distance-Based Algorithms

Scaling changes the coordinate system before distances are calculated. It does not create new information, but it changes how strongly each feature influences an algorithm.

### Standardization

$$
z_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j}.
$$

Each feature is centred around zero and measured in standard-deviation units. It is suitable when mean and standard deviation are representative. It preserves the shape of a distribution but is sensitive to extreme values. It is commonly used before K-Means, PCA, SVM, and nearest-neighbour methods.

### Min–Max scaling

$$
z_{ij}=\frac{x_{ij}-\min(x_j)}{\max(x_j)-\min(x_j)}.
$$

Training values are mapped to $[0,1]$. The bounded range is convenient for neural networks and distance comparisons, but one extreme minimum or maximum can compress most observations into a small interval. New values may also fall outside $[0,1]$.

### Robust scaling

$$
z_{ij}=\frac{x_{ij}-\operatorname{median}(x_j)}{Q_{3j}-Q_{1j}}.
$$

Median and interquartile range are less affected by extreme observations. This is useful for skewed sensor or financial data. However, the output is not bounded, and the method may be unstable for a nearly constant feature whose IQR is close to zero.

### Quantile transformation

The empirical cumulative distribution function is used to map values to ranks:

$$
u_{ij}=\widehat F_j(x_{ij}).
$$

The ranks can remain approximately uniform or be mapped to a normal distribution through $z_{ij}=\Phi^{-1}(u_{ij})$. This transformation limits the influence of extreme magnitudes and can handle strong skewness. Its disadvantage is that it changes spacing and linear relationships, depends on the training distribution, and may map several tied values together.

### Why unscaled features distort Euclidean distance

Suppose meaningful standardized coordinates are $z_j$, but feature $j$ is recorded using scale factor $s_j$, so $x_j=s_jz_j$. Euclidean distance in the recorded units becomes

$$
d_x^2(x_a,x_b)=\sum_{j=1}^{d}s_j^2(z_{aj}-z_{bj})^2.
$$

Therefore, the contribution of feature $j$ is multiplied by $s_j^2$. A feature measured in thousands can dominate one measured in tens even when its standardized difference is less important. The distortion follows directly from the squared scale factor; it is not merely a visual effect.

### Self-constructed proof example

Consider two customers:

| Customer | Annual income (PKR) | Age (years) |
|---|---:|---:|
| A | 50,000 | 20 |
| B | 52,000 | 60 |

The raw Euclidean distance is

$$
\sqrt{(52,000-50,000)^2+(60-20)^2}
=\sqrt{4,001,600}\approx2000.4.
$$

Income contributes $4,000,000/4,001,600\approx99.96\%$ of the squared distance. Now assume the training-set standard deviations are PKR 10,000 for income and 20 years for age. The standardized differences are

$$
\Delta z_{income}=\frac{2000}{10,000}=0.2,
\qquad
\Delta z_{age}=\frac{40}{20}=2.
$$

The standardized distance is

$$
\sqrt{0.2^2+2^2}=2.010.
$$

Age now contributes $4/4.04\approx99.01\%$ because the age difference is two standard deviations while the income difference is only 0.2 standard deviations. This is the comparison the raw units concealed.

### Self-constructed transformation example

For the feature values $[1,2,3,4,100]$:

- Min–Max scaling produces approximately $[0,0.010,0.020,0.030,1]$.
- Standardization uses $\mu=22$ and population $\sigma\approx39.013$, so 100 becomes $(100-22)/39.013\approx1.999$.
- Robust scaling uses median $3$ and IQR $4-2=2$, so 100 becomes $(100-3)/2=48.5$.
- A uniform quantile mapping uses ranks approximately $[0,0.25,0.50,0.75,1]$.

Robust scaling prevents 100 from shifting the centre and IQR, but it does not make the outlier numerically small. Quantile transformation has the strongest effect on spacing because it uses order instead of original gaps.

### Practical conclusion

No scaler is universally best. The correct choice must be tested through downstream clustering metrics. Standardization is a reasonable baseline, RobustScaler is useful under genuine heavy tails, Min–Max scaling is useful when bounded inputs matter, and a quantile transform is justified when rank structure matters more than original spacing.


## 1.1.5 Curse of Dimensionality and Distance Concentration

The curse of dimensionality describes several difficulties that appear as the number of features grows. Space expands so quickly that a fixed amount of data becomes sparse, local neighbourhoods require more observations, and distances can lose their ability to distinguish near points from far points.

### Geometric intuition

Divide each feature axis into ten intervals. Covering a one-dimensional interval requires $10$ cells, but covering a $d$-dimensional unit cube at the same resolution requires

$$
10^d
$$

cells. At $d=2$ this is 100, while at $d=20$ it is $10^{20}$. A dataset that looks large in two dimensions can therefore be extremely sparse in a high-dimensional space.

Another view uses the volume of a $d$-dimensional ball of radius $r$:

$$
V_d(r)=\frac{\pi^{d/2}}{\Gamma(d/2+1)}r^d.
$$

For $r<1$, the factor $r^d$ decreases exponentially with $d$. A fixed-radius neighbourhood contains an increasingly small proportion of the surrounding space.

### Derivation of distance concentration

Let two points $X$ and $Y$ be drawn independently from a $d$-dimensional unit cube. For one coordinate, define

$$
Z_j=(X_j-Y_j)^2.
$$

For independent $X_j,Y_j\sim U(0,1)$,

$$
E[Z_j]=\frac{1}{6},
\qquad
E[Z_j^2]=E[(X_j-Y_j)^4]=\frac{1}{15},
$$

so

$$
\operatorname{Var}(Z_j)=\frac{1}{15}-\frac{1}{36}=\frac{7}{180}.
$$

The squared Euclidean distance is

$$
D^2=\sum_{j=1}^{d}Z_j.
$$

Independence gives

$$
E[D^2]=\frac{d}{6},
\qquad
\operatorname{Var}(D^2)=\frac{7d}{180}.
$$

Its coefficient of variation is therefore

$$
\frac{\sqrt{\operatorname{Var}(D^2)}}{E[D^2]}
=\frac{\sqrt{7d/180}}{d/6}
=\frac{\sqrt{1.4}}{\sqrt d}.
$$

The absolute distance increases with dimension, but its *relative* spread decreases at rate $1/\sqrt d$. Under these assumptions, pairwise distances concentrate around a common scale and the nearest-to-farthest contrast tends to weaken.

### Assumptions and limitations of the derivation

The calculation assumes independent, uniformly distributed coordinates. Real data may be correlated, sparse, or concentrated on a manifold, so the exact numbers will change. The general warning still matters: irrelevant and noisy dimensions can make neighbourhood and density estimates unreliable. High dimensionality is not automatically harmful when the data has strong low-dimensional structure or when an appropriate similarity measure is used.

### Self-constructed numerical example

Using the derived expression:

| Dimension $d$ | $E[D^2]=d/6$ | Relative spread $\sqrt{1.4/d}$ |
|---:|---:|---:|
| 2 | 0.333 | 0.837 |
| 20 | 3.333 | 0.265 |
| 200 | 33.333 | 0.0837 |

The expected squared distance becomes larger, yet its relative variability drops from about $83.7\%$ to $8.37\%$. This helps explain why distance-based clustering may struggle to tell genuinely close and far samples apart when many irrelevant features are present.

### Practical value, limitations, and applications

- **Practical value:** the result motivates feature selection, dimensionality reduction, regularized covariance estimation, and careful metric choice.
- **Limitation:** dimensionality alone does not determine difficulty; intrinsic dimension and data geometry also matter.
- **Applications:** high-dimensional gene expression, image pixels, embeddings, and sensor arrays should be checked for distance concentration before relying on nearest-neighbour or density-based methods.


## 1.1.6 Cluster Validation: Internal, External, and Relative Criteria

Cluster validation asks whether a partition is useful and whether one clustering choice is better than another. No single metric provides a complete answer because different metrics reward different cluster shapes.

### Internal validation

Internal criteria use only features and predicted cluster assignments.

**Silhouette coefficient.** For sample $i$, let $a(i)$ be its mean distance to other members of its own cluster and $b(i)$ the smallest mean distance to another cluster:

$$
s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}}.
$$

Values near 1 indicate good separation, values near 0 indicate a boundary point, and negative values suggest a possibly incorrect assignment. Silhouette analysis is intuitive but depends strongly on the distance measure and often favours compact groups.

**Davies–Bouldin index.** If $S_r$ measures scatter within cluster $r$ and $M_{rs}$ is the distance between cluster centres,

$$
DB=\frac{1}{k}\sum_{r=1}^{k}\max_{s\ne r}\frac{S_r+S_s}{M_{rs}}.
$$

Lower is better. It is efficient and scale-free as a ratio, but centre-based scatter may represent non-spherical clusters poorly.

**Calinski–Harabasz index.** Using between-cluster dispersion $B_k$ and within-cluster dispersion $W_k$,

$$
CH=\frac{\operatorname{tr}(B_k)/(k-1)}{\operatorname{tr}(W_k)/(n-k)}.
$$

Higher is better. It rewards compact, separated clusters, but can prefer convex groups and may grow with the number of samples.

**Dunn index.** A common version is

$$
Dunn=\frac{\min_{r\ne s}\delta(C_r,C_s)}{\max_t\Delta(C_t)},
$$

where $\delta$ is inter-cluster separation and $\Delta$ is cluster diameter. Higher is better. It captures a worst-case separation-to-diameter ratio but is sensitive to outliers and to the exact definitions of $\delta$ and $\Delta$.

**Inertia.** For centroid-based clustering,

$$
J=\sum_{r=1}^{k}\sum_{x_i\in C_r}\|x_i-\mu_r\|^2.
$$

Lower inertia means greater compactness. However, it always decreases as $k$ increases, so it cannot select $k$ by itself.

### External validation

External criteria compare predicted clusters with reference labels that were not used for clustering.

- **Adjusted Rand Index (ARI)** compares agreement between all pairs of samples and corrects the Rand Index for chance. Its adjusted form is

$$
ARI=\frac{RI-E[RI]}{\max(RI)-E[RI]}.
$$

- **Normalized Mutual Information (NMI)** measures shared information between true labels $Y$ and clusters $C$:

$$
NMI(Y,C)=\frac{I(Y;C)}{\sqrt{H(Y)H(C)}}
$$

for one common normalization.
- **Fowlkes–Mallows Index (FMI)** is the geometric mean of pairwise precision and recall:

$$
FMI=\frac{TP}{\sqrt{(TP+FP)(TP+FN)}}.
$$

External metrics help determine whether clusters match known categories, but the known categories may not be the only valid grouping. A low external score does not automatically mean that the clustering is useless.

### Relative validation

Relative validation compares alternative clusterings rather than judging one partition in isolation. Examples include comparing values of $k$, algorithms, scalers, representations, or bootstrap samples. A defensible selection should consider several metrics, stability, computation time, and domain meaning instead of choosing the largest single score.

### Assumptions, advantages, and limitations

| Validation type | Assumption | Advantage | Limitation |
|---|---|---|---|
| Internal | Compactness and separation represent quality | No labels required | Can favour particular shapes or densities |
| External | Reference labels represent the desired grouping | Direct comparison with known outcomes | Requires labels and may reject a valid alternative structure |
| Relative | Candidate methods are evaluated under a fair, repeated setup | Supports model and parameter selection | Only identifies the best candidate among those tested |

Applications include selecting the number of customer segments, checking whether biological clusters match known cell types, and comparing preprocessing pipelines.

### Self-constructed numerical example

Consider the one-dimensional clusters

$$
C_1=\{0,1\},\qquad C_2=\{5,6\}.
$$

For point 0, $a(0)=1$ and $b(0)=(5+6)/2=5.5$, so

$$
s(0)=\frac{5.5-1}{5.5}=0.818.
$$

The four silhouette values are approximately $[0.818,0.778,0.778,0.818]$, giving mean silhouette $0.798$.

The centroids are $0.5$ and $5.5$, so total inertia is

$$
(0-0.5)^2+(1-0.5)^2+(5-5.5)^2+(6-5.5)^2=1.
$$

The overall mean is 3. Between-cluster dispersion is

$$
B=2(0.5-3)^2+2(5.5-3)^2=25,
$$

and $W=1$, so

$$
CH=\frac{25/(2-1)}{1/(4-2)}=50.
$$

Using mean absolute distance to each centroid, both within-cluster scatters are $0.5$ and the centroid distance is 5, giving

$$
DB=\frac{0.5+0.5}{5}=0.2.
$$

The minimum separation between clusters is 4 and the largest cluster diameter is 1, so $Dunn=4$. If reference labels are exactly $[A,A,B,B]$, then ARI, NMI, and FMI all equal 1. The example has excellent scores because it was deliberately constructed with two compact, clearly separated groups.


## 1.1.7 Statistical Significance of Clustering Results

A difference between two clustering scores may be caused by sampling variation. Resampling methods estimate whether an observed result is stable enough to support a conclusion.

### Bootstrap resampling

Given data $X=\{x_1,\ldots,x_n\}$:

1. Draw $n$ observations with replacement to create bootstrap sample $X^{*(b)}$.
2. Refit the complete clustering pipeline on $X^{*(b)}$.
3. Calculate the statistic $T^{*(b)}$, such as mean silhouette or agreement with the original partition.
4. Repeat for $b=1,\ldots,B$.

The percentile confidence interval is

$$
\left[T^*_{(\alpha/2)},\;T^*_{(1-\alpha/2)}\right].
$$

Bootstrap analysis assumes that the observed sample reasonably represents the population. Clustering adds complications: cluster numbers can change, cluster labels can permute, and duplicate observations may affect density methods. Labels should therefore be aligned when necessary, and the full preprocessing and fitting procedure should be repeated inside every resample.

### Permutation test

A permutation test builds a null distribution by destroying the relationship being tested while preserving other data properties. For example, to test whether predicted clusters agree with known classes, randomly permute the known labels and recalculate ARI. If a larger statistic is better, a finite-sample corrected p-value is

$$
p=\frac{1+\sum_{b=1}^{B}\mathbb{1}(T_b^{perm}\geq T_{obs})}{B+1}.
$$

The result is valid only if permutations are exchangeable under the null hypothesis. Ordinary row permutations are inappropriate for strongly time-dependent or grouped observations; block or group-aware permutations may be needed.

### Advantages, limitations, and applications

- **Advantages:** fewer distributional assumptions, direct uncertainty estimates, and a way to distinguish stable improvements from accidental ones.
- **Limitations:** computationally expensive because the pipeline is refitted many times; an invalid resampling scheme produces invalid inference.
- **Applications:** comparing two clustering pipelines, estimating metric confidence intervals, evaluating stability, and testing cluster agreement with partial ground truth.

### Self-constructed numerical example

Suppose a clustering pipeline has observed mean silhouette $T_{obs}=0.52$. After 1,000 bootstrap refits, the 2.5th and 97.5th percentiles of the scores are 0.43 and 0.57. The estimated 95% confidence interval is therefore

$$
[0.43,0.57].
$$

Now suppose the clustering has observed $ARI=0.62$ against partial labels. In 999 random label permutations, only 7 permuted ARI values are at least 0.62. Then

$$
p=\frac{1+7}{999+1}=0.008.
$$

At a 5% significance level, this is evidence that the agreement is stronger than expected from random label alignment. It does not prove that the clusters are the only meaningful grouping; it answers the narrower question defined by the permutation null hypothesis.


# 1.2 Anomaly and Novelty Detection Theory

## 1.2.1 Point, Contextual, and Collective Anomalies

An anomaly is an observation or pattern whose behaviour is sufficiently inconsistent with the chosen definition of normality. “Sufficiently inconsistent” must be defined relative to a domain, context, and acceptable false-alarm rate.

### Point anomaly

A single observation is anomalous when it is unusual by itself. If $s(x)$ is an anomaly score, the decision can be written as

$$
x\text{ is anomalous if }s(x)>\tau,
$$

where $\tau$ is a threshold. This view assumes that individual features contain enough evidence. It is simple and useful for isolated fraud or sensor spikes, but it can miss patterns whose individual values look normal.

### Contextual anomaly

Write a sample as $(c,b)$, where $c$ contains contextual variables such as time, season, or location and $b$ contains behavioural variables. The behaviour is anomalous when

$$
P(B=b\mid C=c)
$$

is unusually small. A temperature of 25°C can be normal in the afternoon but suspicious inside a refrigerated unit at night. Contextual detection is more precise when normal behaviour changes with context, but it requires the correct context variables and enough observations for each context.

### Collective anomaly

A collection or sequence $S=\{x_t,\ldots,x_{t+m}\}$ is anomalous when the joint pattern is unusual even though each member may be acceptable:

$$
P(X_t=x_t,\ldots,X_{t+m}=x_{t+m})<\varepsilon.
$$

This is important for network-event sequences, repeated transactions, and industrial time series. It requires modelling order or dependence; an ordinary independent-row detector may miss it.

### Assumptions, benefits, and limitations

| Type | Main assumption | Benefit | Limitation | Application |
|---|---|---|---|---|
| Point | An individual sample contains enough abnormal evidence | Simple scoring and explanation | Misses context and sequences | Unusually large transaction |
| Contextual | Normal behaviour depends on an observed context | Reduces context-dependent false alarms | Incorrect or missing context misleads the detector | Seasonal energy usage |
| Collective | Dependence between several observations matters | Finds abnormal patterns of normal-looking events | Requires sequence/group modelling | Machine operation cycles |

### Self-constructed numerical examples

1. **Point:** normal sensor values have mean 10 and standard deviation 1. A value of 15 has $z=(15-10)/1=5$, so it is an extreme point anomaly under a three-standard-deviation rule.
2. **Contextual:** suppose daytime temperature has mean 25°C and night temperature has mean 15°C with night standard deviation 2°C. A reading of 25°C is normal during the day but at night has $z=(25-15)/2=5$, making it a contextual anomaly.
3. **Collective:** assume a binary alarm is active in any one second with probability 0.1 and independent activations are the normal model. One active value is not surprising, but six consecutive active values have probability $0.1^6=0.000001$. The run is collectively unusual even though every member is a valid binary value.


## 1.2.2 Density-Based, Distance-Based, and Model-Based Detection

These families differ in how they represent normality.

### Density-based detection

Density-based methods estimate whether a sample lies in a region of low probability density. A simple kernel estimate is

$$
\widehat p(x)=\frac{1}{nh^d}\sum_{i=1}^{n}K\left(\frac{x-x_i}{h}\right).
$$

A low $\widehat p(x)$ implies a high anomaly score. Local methods compare the density around $x$ with densities around its neighbours, which helps when legitimate groups have different densities. These methods assume that local neighbourhoods are meaningful. They can detect irregular shapes, but bandwidth or neighbourhood selection is difficult and density estimation deteriorates in high dimensions.

### Distance-based detection

A distance-based score can use the distance to the $k$th nearest neighbour,

$$
s_k(x)=d\big(x,NN_k(x)\big),
$$

or the mean distance to the $k$ nearest neighbours. Large values indicate isolation. This is intuitive and does not require a full probability model. Its limitations are scale sensitivity, computational cost, and loss of distance contrast in high dimensions.

### Model-based detection

Model-based methods fit a description of normal data or of the full data-generating process. A probabilistic score may be

$$
s(x)=-\log p_\theta(x),
$$

while an autoencoder may use reconstruction error

$$
s(x)=\|x-g_\theta(f_\theta(x))\|_2^2.
$$

These approaches can represent global structure and complex relationships. Their success depends on model assumptions and training quality. A flexible model may even learn to reconstruct anomalies, while a restrictive model may incorrectly flag valid rare cases.

### Comparison

| Family | Assumption | Advantage | Limitation | Suitable use |
|---|---|---|---|---|
| Density-based | Anomalies occupy lower-density neighbourhoods | Can identify local irregularity | Difficult density estimation in high dimensions | Spatial or mixed-density data |
| Distance-based | Anomalies are far from normal neighbours | Transparent score | Metric and scaling strongly affect results | Moderate-dimensional tabular data |
| Model-based | A fitted model represents normal structure | Can capture multivariate relationships | Model misspecification creates false alarms | Images, signals, process monitoring |

### Self-constructed numerical example

Consider the one-dimensional values

$$
[0,0.1,0.2,0.25,3.0].
$$

For $x=0.1$, its two closest neighbours are 0 and 0.2, giving mean neighbour distance

$$
(0.1+0.1)/2=0.1.
$$

For $x=3.0$, the two closest values are 0.25 and 0.2, giving

$$
(2.75+2.8)/2=2.775.
$$

Thus, a distance-based score clearly isolates 3.0. A simple inverse-distance density proxy gives $1/0.1=10$ near 0.1 but only $1/2.775\approx0.36$ near 3.0, so a density method reaches the same conclusion.

If the first four values define a Gaussian normal model, their mean is $0.1375$ and population standard deviation is approximately $0.096$. The model-based z-score of 3.0 is

$$
z=\frac{3.0-0.1375}{0.096}\approx29.8.
$$

All three families detect the constructed anomaly, but they justify the decision differently.


## 1.2.3 Novelty Detection versus Outlier Detection

The terms are related but describe different training conditions.

### Formal distinction

**Novelty detection** assumes that the training set contains normal observations drawn from $P_N(X)$. The model learns a normal acceptance region $A$ such that

$$
P_N(X\in A)\geq1-\alpha,
$$

and a future sample outside $A$ is treated as novel. The training data is expected to be clean or nearly clean.

**Outlier detection** receives a training dataset that may already be contaminated:

$$
P_{train}=(1-\epsilon)P_N+\epsilon P_A,
$$

where $P_A$ is an anomalous distribution and $\epsilon>0$ is the contamination proportion. The task is to identify unusual observations within this mixed dataset.

### When assumptions are violated

- A novelty detector's assumption is violated when anomalies contaminate its normal training set. The learned boundary may expand and accept future faults.
- An outlier detector can fail when contamination is too high, when anomalies form a dense group, or when a small legitimate class is mistaken for contamination.
- Both approaches fail if normal behaviour changes over time and the model is not updated.

### Advantages, limitations, and applications

| Setting | Advantage | Limitation | Example application |
|---|---|---|---|
| Novelty detection | A clean normal class gives a focused boundary | Clean training data may be difficult to guarantee | Detecting new machine faults after training on healthy operation |
| Outlier detection | Can examine an already mixed historical dataset | Must separate rare valid cases from true abnormalities | Auditing past transactions for suspicious records |

### Self-constructed numerical example

Suppose clean normal training measurements are

$$
[9,10,10,11].
$$

Their mean is 10 and population standard deviation is $\sqrt{0.5}=0.707$. A future measurement of 14 has

$$
z=\frac{14-10}{0.707}\approx5.66,
$$

so a novelty detector using a three-standard-deviation boundary rejects it.

Now include 14 in the training data: $[9,10,10,11,14]$. The mean becomes 10.8 and the population standard deviation becomes approximately 1.72. The same value now has

$$
z=\frac{14-10.8}{1.72}\approx1.86.
$$

A non-robust “clean normal” model may accept it. This small example demonstrates why contamination violates the novelty-detection assumption and why robust outlier methods are needed for mixed training data.


# 1.3 Semi-Supervised Learning Theory

## 1.3.1 Self-Training and Pseudo-Labeling

Self-training begins with a supervised model trained on the small labeled set. The model predicts labels for unlabeled samples, adds sufficiently confident predictions to the labeled set as pseudo-labels, and then retrains.

### Formal procedure

At iteration $t$, fit

$$
f^{(t)}=\operatorname{Train}(X_L^{(t)},Y_L^{(t)}).
$$

For an unlabeled sample $x$, define the predicted class and confidence as

$$
\widehat y=\arg\max_c P_{f^{(t)}}(Y=c\mid x),
\qquad
q(x)=\max_c P_{f^{(t)}}(Y=c\mid x).
$$

If $q(x)\geq\tau$, add $(x,\widehat y)$ to the labeled set. The confidence threshold $\tau$ controls a quantity–quality trade-off: a high threshold adds fewer but usually safer labels, while a low threshold adds more labels with greater error risk.

### Convergence behaviour

If pseudo-labels are permanently added, the labeled set grows monotonically and a finite unlabeled pool means the procedure must eventually stop when no sample crosses the threshold or all samples are added. This is algorithmic termination, not a guarantee of convergence to the true classifier. The training objective can stabilize around an incorrect labeling.

### Confirmation bias

An early incorrect prediction is treated as truth during later training. The retrained model may become more confident in the same error and assign similar samples incorrectly. This feedback loop is called confirmation bias. Calibrated probabilities, class-balanced selection, high initial thresholds, soft labels, and validation on genuine labels can reduce but not completely remove the risk.

### Assumptions, advantages, limitations, and applications

- **Assumptions:** the initial labeled set represents every relevant class; confidence is meaningful; and the model's high-confidence errors are rare.
- **Advantages:** simple, model-agnostic, and able to use large unlabeled pools.
- **Limitations:** sensitive to calibration, class imbalance, distribution shift, and early mistakes.
- **Applications:** document classification, image labeling, fault classification, and any setting where obtaining labels is expensive.

### Self-constructed numerical example

Assume a binary classifier gives the following probabilities for class 1:

| Unlabeled sample | $P(Y=1\mid x)$ | True class, hidden from training |
|---|---:|---:|
| $u_1$ | 0.96 | 1 |
| $u_2$ | 0.82 | 0 |
| $u_3$ | 0.55 | 1 |

With $\tau=0.90$, only $u_1$ is added with pseudo-label 1, and that label is correct. With $\tau=0.80$, both $u_1$ and $u_2$ are added as class 1, but $u_2$ is wrong. Suppose retraining on these pseudo-labels raises $P(Y=1\mid u_3)$ from 0.55 to 0.71. The model has moved more unlabeled data toward class 1 partly because it trained on its own mistake. This is a small numerical illustration of confirmation bias rather than genuine new evidence.


## 1.3.2 Co-Training and the Independence Assumption

Co-training represents each sample through two feature views,

$$
x=(x^{(1)},x^{(2)}),
$$

and trains one classifier per view. Each classifier contributes its most confident pseudo-labels to the training data used by the other classifier.

### Core assumptions

The classical formulation makes two strong assumptions:

1. **Sufficiency:** either view contains enough information to predict the label.
2. **Conditional independence:** after conditioning on the label, the two views are independent:

$$
P(X^{(1)},X^{(2)}\mid Y)
=P(X^{(1)}\mid Y)P(X^{(2)}\mid Y).
$$

The intuition is that an error made from one view should not be repeated systematically by the other. Real datasets rarely satisfy perfect independence, but co-training may still help when the views are genuinely different and their errors are not too correlated.

### Procedure

1. Train $f_1$ on view 1 and $f_2$ on view 2 using the labeled set.
2. Let each model predict the unlabeled pool.
3. Select a small, class-balanced set of high-confidence predictions from each model.
4. Give predictions from $f_1$ to the training set of $f_2$, and predictions from $f_2$ to the training set of $f_1$.
5. Repeat until no reliable candidates remain or a maximum number of rounds is reached.

### Advantages, limitations, and applications

- **Advantages:** disagreement between complementary views can create useful new labels; each model teaches the other using different evidence.
- **Limitations:** arbitrary feature splitting is not valid co-training. If both views depend on the same underlying signal, their errors can reinforce one another. Both views must also be sufficiently predictive.
- **Applications:** web pages represented by page text and incoming-link text, audio-visual classification, and industrial systems where analogue and digital sensor groups form defensible views.

### Self-constructed numerical example

Consider an email dataset with two views: word features and sender/link metadata.

| Email | Text-view model | Metadata-view model |
|---|---|---|
| $u_1$ | Spam, 0.95 | Spam, 0.60 |
| $u_2$ | Legitimate, 0.55 | Legitimate, 0.94 |
| $u_3$ | Spam, 0.58 | Legitimate, 0.57 |

At threshold $0.90$, the text model contributes $u_1$ as a spam example to the metadata model. The metadata model contributes $u_2$ as a legitimate example to the text model. Neither model contributes $u_3$. The two high-confidence decisions come from different evidence sources, which is the intended benefit of co-training. If text and metadata were duplicate copies of the same features, the example would violate the independence idea and provide little protection against shared errors.


## 1.3.3 Label Propagation, Label Spreading, and the Graph Laplacian

Graph-based semi-supervised methods treat every sample as a node. Similar samples are connected by weighted edges, and labels are encouraged to vary smoothly across strong edges.

### Building the graph

A common similarity weight is

$$
w_{ij}=\exp\left(-\frac{\|x_i-x_j\|^2}{2\sigma^2}\right),
$$

usually retained only for nearest neighbours. Let $W=[w_{ij}]$ be the adjacency matrix and let the degree matrix be

$$
D_{ii}=\sum_jw_{ij}.
$$

The unnormalized graph Laplacian is

$$
L=D-W.
$$

### Deriving the Laplacian smoothness objective

Let row $f_i$ of matrix $F$ contain the class scores for node $i$. Smooth predictions minimize

$$
\frac{1}{2}\sum_{i,j}w_{ij}\|f_i-f_j\|^2.
$$

Expanding the squared term gives

$$
\begin{aligned}
\frac{1}{2}\sum_{i,j}w_{ij}(f_i^Tf_i+f_j^Tf_j-2f_i^Tf_j)
&=\sum_iD_{ii}f_i^Tf_i-\sum_{i,j}w_{ij}f_i^Tf_j\\
&=\operatorname{tr}(F^T(D-W)F)\\
&=\operatorname{tr}(F^TLF).
\end{aligned}
$$

This derivation shows that the graph Laplacian measures lack of smoothness across connected nodes.

### Label propagation

Label propagation usually **clamps** known labels: their scores remain fixed while unlabeled scores are repeatedly replaced by weighted neighbour averages. Partitioning the Laplacian into labeled and unlabeled blocks gives the harmonic solution

$$
F_U=-L_{UU}^{-1}L_{UL}F_L,
$$

provided the required inverse exists and each unlabeled component is connected to a labeled node.

### Label spreading

Label spreading uses a normalized similarity matrix such as

$$
S=D^{-1/2}WD^{-1/2}
$$

and soft clamping:

$$
F^{(t+1)}=\alpha SF^{(t)}+(1-\alpha)Y,
\qquad 0<\alpha<1.
$$

Because $(1-\alpha)Y$ is reintroduced at every step, original labels influence the solution without being absolutely fixed. This can be more tolerant of label noise, although the choice of $\alpha$ becomes important.

### Assumptions, advantages, limitations, and applications

- **Assumptions:** graph edges represent semantic similarity, labels change smoothly, and every relevant component receives enough labeled information.
- **Advantages:** non-linear boundaries emerge from data geometry; unlabeled samples directly shape the solution.
- **Limitations:** graph construction and storage can be expensive, high-dimensional distances can be unreliable, and a wrong edge can spread a wrong label widely.
- **Applications:** document graphs, image collections, biological networks, and activity-recognition data.

### Self-constructed numerical example

Consider a four-node chain with unit edge weights:

$$
1\;--\;2\;--\;3\;--\;4.
$$

Let node 1 have class score $f_1=0$ and node 4 have $f_4=1$. Nodes 2 and 3 are unlabeled. Minimizing

$$
(f_1-f_2)^2+(f_2-f_3)^2+(f_3-f_4)^2
$$

gives the derivative equations

$$
2f_2-f_1-f_3=0,
\qquad
2f_3-f_2-f_4=0.
$$

Substituting $f_1=0$ and $f_4=1$ gives

$$
f_2=\frac{1}{3},\qquad f_3=\frac{2}{3}.
$$

With threshold 0.5, node 2 receives the class of node 1 and node 3 receives the class of node 4. The linearly changing scores are the smoothest values compatible with the two fixed endpoints.


## 1.3.4 Semi-Supervised Clustering Constraints

Semi-supervised clustering uses limited supervision without requiring a class label for every observation. Pairwise constraints are a common form of supervision.

### Formal definitions

A **must-link** constraint $(i,j)\in\mathcal{M}$ requires

$$
c_i=c_j,
$$

where $c_i$ is the cluster assigned to sample $i$.

A **cannot-link** constraint $(i,j)\in\mathcal{C}$ requires

$$
c_i\ne c_j.
$$

Constrained K-Means, for example, minimizes the usual within-cluster sum of squares subject to

$$
c_i=c_j\;\forall(i,j)\in\mathcal M,
\qquad
c_i\ne c_j\;\forall(i,j)\in\mathcal C.
$$

Hard constraints must be satisfied. Soft constraints instead add a penalty for violations, allowing a solution when supervision is noisy or contradictory.

### Assumptions

- The provided constraints are mostly correct and relevant to the desired grouping.
- Enough constraints connect important regions of the dataset.
- The set of hard constraints is feasible; for example, a pair cannot be both must-link and cannot-link after transitive closure.

### Advantages, limitations, and applications

- **Advantages:** uses simple expert feedback, can separate geometrically overlapping groups, and can give clusters clearer domain meaning.
- **Limitations:** incorrect constraints can severely distort the partition; finding a feasible assignment may be harder; and pairwise labels can still be expensive at scale.
- **Applications:** record linkage, document organization, patient grouping, image organization, and industrial-state clustering.

### Self-constructed numerical example

Take four points:

$$
A=(0,0),\;B=(0.2,0.1),\;C=(3,3),\;D=(3.1,3.2).
$$

With $k=2$, unconstrained K-Means naturally produces $\{A,B\}$ and $\{C,D\}$. Their total within-cluster squared error is only

$$
SSE=0.05.
$$

Now impose must-link $(B,C)$ and cannot-link $(A,B)$. A feasible partition is $\{A\}$ and $\{B,C,D\}$. The centroid of the second cluster is $(2.1,2.1)$, so its SSE is

$$
\begin{aligned}
&\|(0.2,0.1)-(2.1,2.1)\|^2
+\|(3,3)-(2.1,2.1)\|^2\\
&\quad+\|(3.1,3.2)-(2.1,2.1)\|^2
=7.61+1.62+2.21=11.44.
\end{aligned}
$$

The constraints override the much more compact geometric solution. If they encode correct domain knowledge, the higher SSE may be acceptable because the desired concept is not purely geometric. If the constraints are wrong, they damage the clustering. This is why the source and reliability of constraints must be discussed.


# Part 1 Summary

The main connection across Part 1 is that unlabeled data is useful only after a meaningful structure has been defined. Distances and scaling define neighbourhoods; cluster, smoothness, and manifold assumptions connect those neighbourhoods to labels; validation and resampling test whether discovered structure is stable; anomaly detection defines departures from normal structure; and semi-supervised methods transfer limited label information through that structure.

Important cautions are:

1. A distance formula is not meaningful until feature scale and data type are considered.
2. Unlabeled data can hurt when its geometry does not follow the label structure.
3. A high clustering score is evidence under a particular criterion, not proof of one true partition.
4. Anomaly status depends on context and on the definition of normal operation.
5. Pseudo-labels and constraints are useful supervision, but incorrect ones can be amplified.
